<a href="https://colab.research.google.com/github/christy5165/Denoising_Autoencoder.ipynb/blob/main/variational%207.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# 1. Reparameterization Trick
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# 2. Build the Encoder
latent_dim = 2
encoder_inputs = keras.Input(shape=(28, 28, 1))
x = layers.Flatten()(encoder_inputs)
x = layers.Dense(128, activation="relu")(x) # Hidden Layer
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

# 3. Build the Decoder
latent_inputs = keras.Input(shape=(latent_dim,))
x = layers.Dense(128, activation="relu")(latent_inputs) # Hidden Layer
decoder_outputs = layers.Dense(784, activation="sigmoid")(x)
decoder_outputs = layers.Reshape((28, 28, 1))(decoder_outputs)
decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")

# 4. Define the VAE Model with Custom Training Step
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(keras.losses.binary_crossentropy(data, reconstruction), axis=(1, 2))
            )
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            total_loss = reconstruction_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        return {"loss": total_loss, "reconstruction_loss": reconstruction_loss, "kl_loss": kl_loss}

# 5. Load Data & Train
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()
mnist_digits = np.concatenate([x_train, x_test], axis=0).astype("float32") / 255
mnist_digits = np.expand_dims(mnist_digits, -1)

vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam())
vae.fit(mnist_digits, epochs=10, batch_size=128)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - kl_loss: 4.9826 - loss: 182.1786 - reconstruction_loss: 177.1960
Epoch 2/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - kl_loss: 5.0252 - loss: 171.0042 - reconstruction_loss: 165.9790
Epoch 3/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - kl_loss: 4.8956 - loss: 170.4086 - reconstruction_loss: 165.5131
Epoch 4/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - kl_loss: 5.1552 - loss: 159.6446 - reconstruction_loss: 154.4894
Epoch 5/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - kl_loss: 4.7696 - loss: 167.7056 - reconstruction_loss: 162.9360
Epoch 6/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - kl_loss: 5.0098 - loss: 154.9022 - reconstruction_loss: 149.8924
Epoch 7/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - kl_loss: 5.1141 - loss: 162.2412 - reconstruction_loss: 157.1271
Epoch 8/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - kl_loss: 4.9380 - loss: 167.1829 - reconstruction_loss